> `oeai_mod_arbor_bronze.ipynb`
> [20251105.1]
> *Arbor Bronze notebook*

In [ ]:
%run ./oeai_mod_arbor_env_var

In [ ]:
# Initialise Logging
oeai.notebook = SimpleNamespace(
    name = "oeai_mod_arbor_bronze.ipynb",
    buildversion = "20251105.1",
    buildtimestamp = "2025-11-05T16:00:00Z",
)
oeai.log.init()
oeai.log.start_block("Notebook Init")

In [ ]:
# Snowflake connection parameters
conn_params = {
    'user': arbor_db_User,
    'password': arbor_db_Pass,
    'account': arbor_db_Account,
    'warehouse': arbor_db_Warehouse,
    'database': arbor_db_Database,
    'schema': arbor_db_Schema,
}

In [ ]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, TimestampNTZType, IntegerType, DateType, BooleanType, FloatType, LongType
from pyspark.sql.types import TimestampType

In [ ]:
# Define schemas for each view
schemas = {
    "STUDENTS": StructType([
        StructField("Student_Unique_ID",StringType(),True),
        StructField("Student_ID",IntegerType(),True),
        StructField("Application_ID",StringType(),False),
        StructField("UPN",StringType(),True),
        StructField("Legal_First_Name",StringType(),True),    
        StructField("Legal_Last_Name",StringType(),True), 
        StructField("Preffered_First_Name",StringType(),True), 
        StructField("Preffered_Last_Name",StringType(),True), 
        StructField("Date_Of_Birth",DateType(),True),
        StructField("Is_Current",BooleanType(),True), 
        StructField("Gender",StringType(),True), 
        StructField("Sex",StringType(),True),
        StructField("Ethnicity",StringType(),True),
        StructField("Ethnicity_Category",StringType(),True),    
        StructField("Ethnicity_Category_Name",StringType(),True),
        StructField("Religion",StringType(),True), 
        StructField("Native_Language_Codes",StringType(),True), 
        StructField("Native_Language_Names",StringType(),True),
        StructField("Email_Address",StringType(),True), 
        StructField("Nationalities",StringType(),True), 
        StructField("Current_Year_Group",StringType(),True), 
        StructField("Current_Curriculum_Grade_Name",StringType(),True),
        StructField("Current_SEN_Status",StringType(),True),
        StructField("Current_Child_Protection_Status_Name",StringType(),True),
        StructField("Young_Carer",BooleanType(),True),
        StructField("Young_Carer_Source",StringType(),True),
        StructField("Current_Registration_Group",StringType(),True),
        StructField("Address",StringType(),True), 
        StructField("PostCode",StringType(),True),
        StructField("Child_Protection",BooleanType(),True),
        StructField("Gifted",BooleanType(),True), 
        StructField("Gifted_Talent",BooleanType(),True),
        StructField("Has_Medical_Condition",BooleanType(),True),    
        StructField("In_Care",BooleanType(),True), 
        StructField("In_Year_Admission",BooleanType(),True), 
        StructField("Out_Of_Age_Cohort",BooleanType(),False),
        StructField("SEN",BooleanType(),True), 
        StructField("Talented",BooleanType(),False),
        StructField("EAL",BooleanType(),True), 
        StructField("Compulsory_School_Age",BooleanType(),True),
        StructField("Ever_6_FSM",BooleanType(),True),
        StructField("Early_Years_Pupil_Premium_Recipient",BooleanType(),True),
        StructField("Ever_6_Service_Child",BooleanType(),False),
        StructField("FSM",BooleanType(),True),  
        StructField("Mobile_Y10_Y11",BooleanType(),True), 
        StructField("Mobile_Y5_Y6",BooleanType(),False),
        StructField("Pupil_Premium",BooleanType(),False),
        StructField("Pupil_Premium_Recipient",BooleanType(),True), 
        StructField("Service_Child",BooleanType(),False), 
        StructField("Traveller",BooleanType(),True), 
        StructField("Disadvantaged",BooleanType(),False),
        StructField("SEN_Support",BooleanType(),False),
        StructField("Education_Health_Care_Plan",BooleanType(),False), 
    ]),
    "STUDENT_SEN_NEEDS": StructType([
        StructField("SEN_Need_Unique_ID",StringType(),True),
        StructField("SEN_Need_ID",IntegerType(),False),
        StructField("Application_ID",StringType(),True),
        StructField("Student_Unique_ID",StringType(),True),
        StructField("Student_ID",IntegerType(),True),
        StructField("SEN_Need_Type",StringType(),True),
        StructField("SEN_Need_Description",StringType(),True),
        StructField("SEN_Need_Ranking",IntegerType(),True),
        StructField("Effective_Date",DateType(),True),
        StructField("End_Date",DateType(),True),
    ]),
    "STUDENT_SCHOOL_ENROLMENTS": StructType([
        StructField("Student_Unique_ID",StringType(),True),
        StructField("Student_School_Enrolment_ID",IntegerType(),False),
        StructField("Student_School_Enrolment_Unique_ID",StringType(),False),
        StructField("Application_ID",StringType(),True),
        StructField("Student_ID",IntegerType(),True),
        StructField("Start_Date",DateType(),True),
        StructField("End_Date",DateType(),True),
        StructField("Currently_On_Roll",BooleanType(),True),
        StructField("Unenrolment_Reason_Code",StringType(),True),
	]),
   "STUDENT_REGISTRATION_FORM_MEMBERSHIPS":StructType([
        StructField("Registration_Form_Membership_Unique_ID",StringType(),True),
        StructField("Student_Unique_ID",StringType(),False),
        StructField("Application_ID",StringType(),True),
        StructField("Student_ID",IntegerType(),True),
        StructField("Registration_Form_ID",IntegerType(),True),
        StructField("Registration_Form_Unique_ID",StringType(),True),
        StructField("Registration_Form_Name",StringType(),True),
        StructField("Registration_Room_Name",StringType(),True),
        StructField("Start_Date",DateType(),True),
        StructField("End_Date",DateType(),True),
   ]),
   "STUDENT_SEN_STATUS_ASSIGNMENT":StructType([
        StructField("Student_Unique_ID",StringType(),True),
        StructField("Application_ID",StringType(),False),
        StructField("Student_ID",IntegerType(),True),
        StructField("Student_SEN_Status_Assignment_ID",IntegerType(),True),
        StructField("Student_SEN_Status_Assignment_Unique_ID",StringType(),True),
        StructField("SEN_Status_Code",StringType(),True),
        StructField("SEN_Status_Name",StringType(),True),
        StructField("Counts_As_SEN_Status",BooleanType(),True),
        StructField("SEN_Status_Export_Code",StringType(),True),
        StructField("Is_Member_Of_SEN_Unit",BooleanType(),True),
        StructField("Has_Resourced_Provision",BooleanType(),True),
        StructField("Start_Date",DateType(),True),
        StructField("End_Date",DateType(),True),
   ]),
   "STUDENT_TAGS_HISTORY":StructType([
        StructField("Student_Unique_ID",StringType(),True),
        StructField("Application_ID",StringType(),True),
        StructField("Student_ID",IntegerType(),True),
        StructField("Student_Tagging_unique_Id",StringType(),True),
        StructField("Tag_Identifier",StringType(),True),
        StructField("Start_Date",DateType(),True),
        StructField("End_Date",DateType(),True),
   ]), 
   "STUDENT_YEAR_GROUP_MEMBERSHIPS":StructType([
        StructField("Year_Group_Membership_Unique_ID",StringType(),True),
        StructField("Student_Unique_ID",StringType(),True),
        StructField("Application_ID",StringType(),True),
        StructField("Student_ID",IntegerType(),True),
        StructField("Year_Group_Unique_ID",StringType(),True),
        StructField("Curriculum_Grade_ID",IntegerType(),True),
        StructField("Curriculum_Grade_Unique_ID",StringType(),True),
        StructField("START_DATE",DateType(),True),
        StructField("END_DATE",DateType(),True),
        StructField("IS_Current",BooleanType(),True),
   ]),
   "YEAR_GROUPS": StructType([
        StructField("Application_ID",StringType(),True),
        StructField("Year_Group_Unique_ID",StringType(),False),
        StructField("Year_Group_ID",IntegerType(),True),
        StructField("Year_Group",StringType(),True),
        StructField("Academic_Level_Name",StringType(),True),
        StructField("Academic_Year_ID",IntegerType(),True),
        StructField("Academic_Year_Unique_ID",StringType(),True),
        StructField("Curriculum_Grade_ID",IntegerType(),True),
        StructField("Curriculum_Grade_Unique_ID",StringType(),True),
        StructField("Curriculum_Grade_Name",StringType(),True),
   ]),
    "SUSPENSIONS" : StructType([
        StructField("SUSPENSION_ID",IntegerType(),True),
        StructField("SUSPENSION_UNIQUE_ID",StringType(),False),
        StructField("STUDENT_UNIQUE_ID",StringType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("STUDENT_ID",IntegerType(),True),
        StructField("START_DATE",TimestampNTZType(),True),
        StructField("END_DATE",TimestampNTZType(),True),
        StructField("DECISION_DATE",TimestampNTZType(),True),
        StructField("SUSPENSION_REASONS",StringType(),True),
        StructField("SUSPENSION_REASONS_LIST",StringType(),True),
        StructField("SUSPENSION_REASONS_NAME",StringType(),True),
        StructField("SUSPENSION_REASONS_NAME_LIST",StringType(),True),
        StructField("DAILY_START_TIME",StringType(),True),
        StructField("DAILY_END_TIME",StringType(),True),
        StructField("STATISTICAL_DAYS_EXCLUDED",FloatType(),True),
        StructField("NARRATIVE",StringType(),True),
        StructField("DFE_EXPORT_CODE",StringType(),True),
        StructField("STATISTICAL_SESSIONS_EXCLUDED",FloatType(),True),
        StructField("SCHOOL_DEFINED_SESSIONS_EXCLUDED",IntegerType(),True),
        StructField("SCHOOL_DEFINED_DAYS_EXCLUDED",FloatType(),True),
   ]),
    "SCHOOLS" : StructType ([
        StructField("Application_ID",StringType(),True),
        StructField("Name",StringType(),False),
        StructField("Abbreviated_Name",StringType(),True),
        StructField("Phase",StringType(),True),
        StructField("URN",StringType(),True),
	]),
    "SITES" : StructType([
        StructField("SITE_ID",IntegerType(),True),
        StructField("SITE_UNIQUE_ID",StringType(),False),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("SITE_NAME",StringType(),True),
	]),
    "PERMANENT_EXCLUSIONS" : StructType([
        StructField("PERMANENT_EXCLUSION_ID",IntegerType(),True),
        StructField("PERMANENT_EXCLUSION_UNIQUE_ID",StringType(),False),
        StructField("STUDENT_UNIQUE_ID",StringType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("STUDENT_ID",IntegerType(),True),
        StructField("START_DATE",TimestampNTZType(),True),
        StructField("DECISION_DATE",TimestampNTZType(),True),
        StructField("EXCLUSION_REASONS",StringType(),True),
        StructField("EXCLUSION_REASONS_LIST",StringType(),True),
        StructField("EXCLUSION_REASONS_NAME",StringType(),True),
        StructField("EXCLUSION_REASONS_NAME_LIST",StringType(),True),
        StructField("CURRICULUM_GRADE_NAME_ON_LEAVING",StringType(),True),
        StructField("DFE_EXPORT_CODE",StringType(),True),
    ]),
    "STUDENT_ACADEMIC_YEAR_ENROLMENTS" :StructType([
        StructField("Student_Unique_ID",StringType(),True),
        StructField("Academic_Year_Enrolment_ID",IntegerType(),False),
        StructField("Academic_Year_Enrolment_Unique_ID",StringType(),True),
        StructField("Application_ID",StringType(),True),
        StructField("Academic_Year_ID",IntegerType(),True),
        StructField("Academic_Year_Unique_ID",StringType(),True),
        StructField("Academic_Year_Name",StringType(),True),
        StructField("Student_ID",IntegerType(),True),
        StructField("Start_Date",DateType(),True),
        StructField("End_Date",DateType(),True),
        StructField("Is_Current",BooleanType(),True),
        StructField("Enrolment_Notes",StringType(),True),
        StructField("Enrolment_Mode",StringType(),True),
	]),
    "ROLL_CALL_ATTENDANCE" :StructType([
        StructField("ATTENDANCE_ROLL_CALL_RECORD_ID",IntegerType(),True),
        StructField("ATTENDANCE_ROLL_CALL_RECORD_UNIQUE_ID",StringType(),True),
        StructField("STUDENT_UNIQUE_ID",StringType(),True),
        StructField("STUDENT_ID",IntegerType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("ATTENDANCE_ROLL_CALL_ID",IntegerType(),True),
        StructField("ATTENDANCE_ROLL_CALL_UNIQUE_ID",StringType(),True),
        StructField("RAW_MARK",StringType(),True),
        StructField("MARK_CODE",StringType(),True),
        StructField("MINUTES_LATE",IntegerType(),True),
        StructField("PERIOD",StringType(),True),
        StructField("DATE",DateType(),True),
        StructField("ACADEMIC_YEAR_UNIQUE_ID",StringType(),True),
        StructField("ACADEMIC_YEAR_ID",IntegerType(),True),
        StructField("ACADEMIC_YEAR_NAME",StringType(),True),
        StructField("IS_PRESENT",BooleanType(),True),
        StructField("IS_AUTHORIZED_ABSENT",BooleanType(),True),
        StructField("IS_UNAUTHORIZED_ABSENT",BooleanType(),True),
        StructField("IS_POSSIBLE_ATTENDANCE",BooleanType(),True),
    ]),
    "INTERNAL_EXCLUSIONS" :StructType([
        StructField("INTERNAL_EXCLUSION_UNIQUE_ID",StringType(),True),
        StructField("INTERNAL_EXCLUSION_ID",IntegerType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("STUDENT_UNIQUE_ID",StringType(),True),
        StructField("STUDENT_ID",IntegerType(),True),
        StructField("ISSUED_DATETIME",TimestampNTZType(),True),
        StructField("ISSUED_BY_STAFF_UNIQUE_ID",StringType(),True),
        StructField("ISSUED_BY_STAFF_ID",StringType(),True),
        StructField("INTERNAL_EXCLUSION_REASONS_NAME",StringType(),True),
        StructField("INTERNAL_EXCLUSION_REASONS_TYPE_NAME",StringType(),True),
        StructField("NARRATIVE",StringType(),True),
        StructField("TIMETABLE_SLOT_UNIQUE_ID",StringType(),True),
        StructField("TIMETABLE_SLOT_ID",FloatType(),True),
        StructField("LOCATION_UNIQUE_ID",StringType(),True),
        StructField("LOCATION_ID",FloatType(),True),
        StructField("SESSION_NAME",StringType(),True),
        StructField("START_DATETIME",TimestampNTZType(),True),
        StructField("END_DATETIME",TimestampNTZType(),True),
        StructField("RAW_MARK",StringType(),True),
        StructField("MARK_CODE",StringType(),True),
        StructField("MINUTES_LATE",FloatType(),True),
        StructField("IS_PRESENT",BooleanType(),True),
        StructField("IS_AUTHORIZED_ABSENT",BooleanType(),True),
        StructField("IS_UNAUTHORIZED_ABSENT",BooleanType(),True),
        StructField("IS_POSSIBLE_ATTENDANCE",BooleanType(),True),
    ]),
    "USER_DEFINED_FIELDS_VALUES" :StructType([
        StructField("USER_DEFINED_FIELD_VALUE_UNIQUE_ID",StringType(),True),
        StructField("USER_DEFINED_FIELD_VALUE_ID",IntegerType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("USER_DEFINED_FIELD_ID",IntegerType(),True),
        StructField("USER_DEFINED_FIELD_UNIQUE_ID",StringType(),True),
        StructField("VALUE",StringType(),True),
        StructField("STUDENT_UNIQUE_ID",StringType(),True),
        StructField("STAFF_UNIQUE_ID",StringType(),True),
        StructField("GUARDIAN_UNIQUE_ID",StringType(),True),
    ]),
    "BEHAVIOURAL_INCIDENTS" :StructType([
        StructField("BEHAVIOURAL_INCIDENT_ID",FloatType(),True),
        StructField("BEHAVIOUR_INCIDENT_UNIQUE_ID",StringType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("BEHAVIOUR_NAME",StringType(),True),
        StructField("SEVERITY",IntegerType(),True),
        StructField("INCIDENT_DATE",TimestampNTZType(),True),
        StructField("RESOLVED_DATE",TimestampNTZType(),True),
        StructField("LOGGED_BY_STAFF_ID",IntegerType(),True),
        StructField("LOGGED_BY_STAFF_UNIQUE_ID",StringType(),True),
        StructField("NARRATIVE",StringType(),True),
        StructField("IS_RESOLVED",BooleanType(),True),
        StructField("BEHAVIOUR_LOCATION_NAME",StringType(),True),
        StructField("ROOM_UNIQUE_ID",StringType(),True),
        StructField("LESSON_UNIQUE_ID",StringType(),True),
        StructField("ACADEMIC_YEAR_UNIQUE_ID",StringType(),True),
        StructField("ACADEMIC_YEAR_ID",IntegerType(),True),
        StructField("ACADEMIC_YEAR_NAME",StringType(),True),
    ]),
    "BEHAVIOURAL_INCIDENT_STUDENT_INVOLVEMENTS" :StructType([
        StructField("BEHAVIOURAL_INCIDENT_INVOLVEMENT_UNIQUE_ID",StringType(),True),
        StructField("BEHAVIOURAL_INCIDENT_INVOLVEMENT_ID",FloatType(),True),
        StructField("BEHAVIOURAL_INCIDENT_ID",FloatType(),True),
        StructField("BEHAVIOURAL_INCIDENT_UNIQUE_ID",StringType(),True),
        StructField("STUDENT_UNIQUE_ID",StringType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("STUDENT_ID",IntegerType(),True),
        StructField("SEVERITY",IntegerType(),True),
        StructField("COMMENT",StringType(),True),
    ]),
    "REGISTRATION_FORMS" :StructType([
        StructField("REGISTRATION_FORM_UNIQUE_ID",StringType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("REGISTRATION_FORM_ID",IntegerType(),True),
        StructField("REGISTRATION_FORM_NAME",StringType(),True),
        StructField("REGISTRATION_FORM_ROOM",IntegerType(),True),
        StructField("ACADEMIC_YEAR_UNIQUE_ID",StringType(),True),
    ]),
    "STUDENT_REGISTRATION_FORM_MEMBERSHIPS" :StructType([
        StructField("REGISTRATION_FORM_MEMBERSHIP_UNIQUE_ID",StringType(),True),
        StructField("STUDENT_UNIQUE_ID",StringType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("STUDENT_ID",IntegerType(),True),
        StructField("REGISTRATION_FORM_ID",IntegerType(),True),
        StructField("REGISTRATION_FORM_UNIQUE_ID",StringType(),True),
        StructField("REGISTRATION_FORM_NAME",StringType(),True),
        StructField("REGISTRATION_FORM_ROOM_NAME",StringType(),True),
        StructField("START_DATE",DateType(),True),
        StructField("END_DATE",DateType(),True),
    ]),
    "USER_DEFINED_FIELDS" :StructType([
        StructField("USER_DEFINED_FIELD_UNIQUE_ID",StringType(),True),
        StructField("USER_DEFINED_FIELD_ID",IntegerType(),True),
        StructField("APPLICATION_ID",StringType(),True),
        StructField("CODE",StringType(),True),
        StructField("FIELD_NAME",StringType(),True),
        StructField("FIELD_TYPE",StringType(),True),
        StructField("IDENTIFIER",StringType(),True),
        StructField("RELATED_TABLE_NAME",StringType(),True),
        StructField("USER_DEFINED_FIELD_SOURCE",StringType(),True),
    ]),
    "STAFF" :StructType([
        StructField("STAFF_UNIQUE_ID", StringType(), True),
        StructField("STAFF_ID", IntegerType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("LEGAL_FIRST_NAME", StringType(), True),
        StructField("LEGAL_LAST_NAME", StringType(), True),
        StructField("PREFERRED_FIRST_NAME", StringType(), True),
        StructField("PREFERRED_LAST_NAME", StringType(), True),
        StructField("DATE_OF_BIRTH", DateType(), True),
        StructField("TITLE", StringType(), True),
        StructField("STAFF_NUMBER", StringType(), True),
        StructField("LEGACY_SYSTEM_ID", StringType(), True),
        StructField("TIMETABLE_ABBREVIATION", StringType(), True),
        StructField("SEX", StringType(), True),
        StructField("GENDER", StringType(), True),
        StructField("BIRTH_COUNTRY", StringType(), True),
        StructField("ETHNICITY", StringType(), True),
        StructField("ETHNICITY_CATEGORY", StringType(), True),
        StructField("ETHNICITY_CATEGORY_NAME", StringType(), True),
        StructField("RELIGION", StringType(), True),
        StructField("CONTINUOUS_SERVICE_START_DATE", TimestampType(), True),
        StructField("QUALIFIED_TEACHER_STATUS", BooleanType(), True),
        StructField("EARLY_YEARS_TEACHER_STATUS", BooleanType(), True),
        StructField("QUALIFIED_TEACHER_LEARNING_SKILLS_STATUS", BooleanType(), True),
        StructField("NEWLY_QUALIFIED_TEACHER_DATE", TimestampType(), True),
        StructField("HLTA_STATUS", BooleanType(), True),
        StructField("IS_TEACHING_STAFF", BooleanType(), True),
        StructField("EMAIL", StringType(), True),
        StructField("ADDRESS_1", StringType(), True),
        StructField("ADDRESS_2", StringType(), True),
        StructField("POSTAL_TOWN", StringType(), True),
        StructField("POSTCODE", StringType(), True),
    ]),
    "STAFF_ABSENCES" :StructType([
        StructField("STAFF_ABSENCE_UNIQUE_ID", StringType(), True),
        StructField("STAFF_ABSENCE_ID", IntegerType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("STAFF_ID", IntegerType(), True),
        StructField("STAFF_UNIQUE_ID", StringType(), True),
        StructField("APPROVED_BY_STAFF_ID", IntegerType(), True),
        StructField("APPROVED_BY_STAFF_UNIQUE_ID", StringType(), True),
        StructField("CALCULATED_WORKING_DAYS", FloatType(), True),
        StructField("ACTUAL_WORKING_DAYS", FloatType(), True),
        StructField("WORKING_DAYS", FloatType(), True),
        StructField("CALCULATED_WORKING_HOURS", FloatType(), True),
        StructField("ACTUAL_WORKING_HOURS", FloatType(), True),
        StructField("WORKING_HOURS", FloatType(), True),
        StructField("START_DATETIME", TimestampType(), True),
        StructField("END_DATETIME", TimestampType(), True),
        StructField("CATEGORY_CODE", StringType(), True),
        StructField("CATEGORY_NAME", StringType(), True),
        StructField("IS_PAID_LEAVE", BooleanType(), True),
        StructField("IS_PAID_HOLIDAY", BooleanType(), True),
        StructField("IS_PAID_MATERNITY", BooleanType(), True),
        StructField("IS_PAID_PATERNITY", BooleanType(), True),
        StructField("IS_AUTHORIZED_ABSENCE", BooleanType(), True),
        StructField("SICKNESS_CATEGORY_CODE", StringType(), True),
        StructField("SICKNESS_CATEGORY_NAME", StringType(), True),
        StructField("SICKNESS_SUBCATEGORY_CODE", StringType(), True),
        StructField("SICKNESS_SUBCATEGORY_NAME", StringType(), True),
    ]),
    "STAFF_CONTRACT" :StructType([
        StructField("STAFF_CONTRACT_UNIQUE_ID", StringType(), True),
        StructField("STAFF_CONTRACT_ID", LongType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("STAFF_ID", LongType(), True),
        StructField("STAFF_UNIQUE_ID", StringType(), True),
        StructField("CONTRACT_NAME", StringType(), True),
        StructField("CONTRACT_REFERENCE", StringType(), True),
        StructField("ISSUED_DATE", DateType(), True),
        StructField("CONTRACT_START_DATE", DateType(), True),
        StructField("CONTRACT_EXPECTED_END_DATE", DateType(), True),
        StructField("CONTRACT_END_DATE", DateType(), True),
        StructField("EMPLOYMENT_TYPE_CODE", StringType(), True),
        StructField("EMPLOYMENT_TYPE_NAME", StringType(), True),
        StructField("STAFF_ORIGIN_CODE", StringType(), True),
        StructField("STAFF_ORIGIN_DESCRIPTION", StringType(), True),
        StructField("STAFF_DESTINATION_CODE", StringType(), True),
        StructField("STAFF_DESTINATION_DESCRIPTION", StringType(), True),
        StructField("STAFF_LEAVING_REASON_CODE", StringType(), True),
        StructField("STAFF_LEAVING_REASON_DESCRIPTION", StringType(), True),
        StructField("STAFF_SUPERANNUATION_SCHEME_CODE", StringType(), True),
        StructField("STAFF_SUPERANNUATION_SCHEME_NAME", StringType(), True),
        StructField("CONTRACT_POST_ID", StringType(), True),  # Using StringType for VARIANT
        StructField("STAFF_CONTRACT_POST_UNIQUE_ID", StringType(), True),
        StructField("CONTRACT_POST_JOB_TITLE", StringType(), True),
        StructField("CONTRACT_POST_POSITION_ID", StringType(), True),  # Using StringType for VARIANT
        StructField("POSITION_NAME", StringType(), True),
        StructField("POSITION_REFERENCE", StringType(), True),
        StructField("EXPECTED_HOURS_PER_WEEK", FloatType(), True),
        StructField("EXPECTED_WEEKS_PER_YEAR", FloatType(), True),
        StructField("FTE_HOURS_PER_WEEK", FloatType(), True),
        StructField("FTE_PROPORTION", FloatType(), True),
        StructField("POSITION_BUSINESS_ROLE_ID", StringType(), True),  # Using StringType for VARIANT
        StructField("POSITION_BUSINESS_ROLE_RANKING", StringType(), True),  # Using StringType for VARIANT
        StructField("WORKFORCE_CENSUS_ROLE_IDENTIFIER", StringType(), True),
        StructField("UKDFE_POSITION_CATEGORY_CODE", StringType(), True),
        StructField("UKDFE_POSITION_CATEGORY_LABEL", StringType(), True),
        StructField("POSITION_START_DATE", TimestampType(), True),
        StructField("POSITION_EXPECTED_END_DATE", TimestampType(), True),
        StructField("BUSINESS_ROLE_UNIQUE_ID", StringType(), True),
        StructField("PAYROLL_NUMBER", StringType(), True)
    ]),
    "STAFF_CONTRACT_POST_SALARY" :StructType([
        StructField("STAFF_CONTRACT_POST_SALARY_UNIQUE_ID", StringType(), True),
        StructField("STAFF_CONTRACT_POST_SALARY_ID", IntegerType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("GROSS_SALARY", StringType(), True),
        StructField("HOURS_PER_WEEK", FloatType(), True),
        StructField("WEEKS_PER_YEAR", FloatType(), True),
        StructField("FTE_HOURS_PER_WEEK", FloatType(), True),
        StructField("FTE_WEEKS_PER_YEAR", FloatType(), True),
        StructField("EFFECTIVE_DATE", TimestampType(), True),
        StructField("END_DATE", TimestampType(), True),
        StructField("SAFEGUARDED_PERIOD_START_DATE", TimestampType(), True),
        StructField("SAFEGUARDED_PERIOD_END_DATE", TimestampType(), True),
        StructField("PAY_SCALE_UNIQUE_ID", StringType(), True),
        StructField("PAY_SCALE_GRADE_UNIQUE_ID", StringType(), True),
        StructField("PAY_SCALE_SPINAL_POINT_UNIQUE_ID", StringType(), True),
        StructField("STAFF_CONTRACT_POST_ID", IntegerType(), True),
        StructField("STAFF_CONTRACT_POST_UNIQUE_ID", StringType(), True)
    ]),
    "STAFF_PAY_SCALES" :StructType([
        StructField("PAY_SCALE_UNIQUE_ID", StringType(), True),
        StructField("PAY_SCALE_ID", IntegerType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("PAY_SCALE_CODE", StringType(), True),
        StructField("PAY_SCALE_NAME", StringType(), True),
        StructField("EFFECTIVE_DATE", TimestampType(), True),
        StructField("END_DATE", TimestampType(), True),
        StructField("MINIMUM_SALARY", FloatType(), True),
        StructField("MAXIMUM_SALARY", FloatType(), True),
        StructField("PAY_SCALE_GRADES_ID", IntegerType(), True),
        StructField("PAY_SCALE_GRADES_GRADE_NAME", StringType(), True),
        StructField("PAY_SCALE_GRADES_DATA_ORDER", IntegerType(), True),
        StructField("PAY_SCALE_SPINAL_POINTS_ID", IntegerType(), True),
        StructField("PAY_SCALE_SPINAL_POINTS_CODE", StringType(), True),
        StructField("PAY_SCALE_SPINAL_POINTS_NAME", StringType(), True),
        StructField("PAY_SCALE_GRADES_UNIQUE_ID", StringType(), True),
        StructField("PAY_SCALE_SPINAL_POINTS_UNIQUE_ID", StringType(), True)
    ]),
    "EXAM_RESULTS" :StructType([
        StructField("QUALIFICATION_RESULT_UNIQUE_ID", StringType(), True),
        StructField("QUALIFICATION_RESULT_ID", LongType(), True),  # LongType for large numbers
        StructField("APPLICATION_ID", StringType(), True),
        StructField("STUDENT_UNIQUE_ID", StringType(), True),
        StructField("STUDENT_ID", LongType(), True),  # LongType for large numbers
        StructField("QUALIFICATION_RESULT_TYPE", StringType(), True),  # Limited to 13 characters
        StructField("SCHEME_CODE", StringType(), True),
        StructField("AWARD_NAME", StringType(), True),
        StructField("QAN", StringType(), True),
        StructField("EXAM_SERIES", StringType(), True),
        StructField("SERIES_START_DATE", TimestampType(), True),
        StructField("GRADE", StringType(), True),
        StructField("NUMERICAL_GRADE", FloatType(), True),
        StructField("GRADE_SHORT_NAME", StringType(), True),
        StructField("MISSING_RESULT_REASON", StringType(), True),
        StructField("RESULT_YEAR", IntegerType(), True)  # 5-digit number, IntegerType should be sufficient
    ]),
    "SUMMATIVE_ASSESSMENT_MARKS" :StructType([
        StructField("APPLICATION_ID", StringType(), True),
        StructField("STUDENT_ID", StringType(), True),
        StructField("STUDENT_UNIQUE_ID", StringType(), True),
        StructField("SUMMATIVE_ASSESSMENT_MARK_ID", LongType(), True),  # Large integer
        StructField("SUMMATIVE_ASSESSMENT_MARK_UNIQUE_ID", StringType(), True),
        StructField("ASSESSMENT_ID", LongType(), True),  # Large integer
        StructField("ASSESSMENT_UNIQUE_ID", StringType(), True),
        StructField("SUBJECT_CODE", StringType(), True),
        StructField("SUBJECT_NAME", StringType(), True),
        StructField("SUBJECT_UNIQUE_ID", StringType(), True),
        StructField("SUBJECT_ID", LongType(), True),  # Large integer
        StructField("ACADEMIC_YEAR_UNIQUE_ID", StringType(), True),
        StructField("ACADEMIC_YEAR_ID", LongType(), True),  # Large integer
        StructField("ACADEMIC_YEAR_START", StringType(), True),
        StructField("ACADEMIC_YEAR_NAME", StringType(), True),
        StructField("ASSESSMENT_CODE", StringType(), True),
        StructField("ASSESSMENT_NAME", StringType(), True),
        StructField("ASSESSMENT_DATE", DateType(), True),
        StructField("PERIOD_NAME", StringType(), True),
        StructField("PERIOD_START", DateType(), True),
        StructField("PERIOD_END", DateType(), True),
        StructField("GRADE_SET_CODE", StringType(), True),
        StructField("GRADE_POINTS", FloatType(), True),
        StructField("GPS_MIN", LongType(), True),  # Large integer
        StructField("GPS_MAX", LongType(), True),  # Large integer
        StructField("GRADE_NAME", StringType(), True),
        StructField("NON_SUBMISSION_REASON", StringType(), True),
        StructField("STAFF_ID", LongType(), True),  # Large integer
        StructField("STAFF_UNIQUE_ID", StringType(), True)
    ]),
    "STANDARDISED_ASSESSMENT_MARKS" :StructType([
        StructField("STUDENT_UNIQUE_ID", StringType(), True),
        StructField("STANDARDISED_ASSESSMENT_MARK_UNIQUE_ID", StringType(), True),
        StructField("STANDARDISED_ASSESSMENT_MARK_ID", LongType(), True),  # Large integer
        StructField("APPLICATION_ID", StringType(), True),
        StructField("STUDENT_ID", LongType(), True),  # Large integer
        StructField("ASSESSMENT_NAME", StringType(), True),
        StructField("ASSESSMENT_CODE", StringType(), True),
        StructField("MARK_GRADE", StringType(), True),
        StructField("MARK_NUMERIC", FloatType(), True),
        StructField("MARK_COMMENT", StringType(), True),  # VARIANT mapped to StringType (alternative could be MapType/StringType JSON)
        StructField("MARK_CATEGORY", StringType(), True),
        StructField("INTENDED_CURRICULUM_GRADE", StringType(), True),
        StructField("ASSESSMENT_DATE", TimestampType(), True),
        StructField("SUBJECT_CODE", StringType(), True),
        StructField("SUBJECT_NAME", StringType(), True)
    ]),
    "LESSON_ATTENDANCE" :StructType([
        StructField("LESSON_ATTENDANCE_RECORD_UNIQUE_ID", StringType(), True),
        StructField("LESSON_ATTENDANCE_RECORD_ID", LongType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("STUDENT_UNIQUE_ID", StringType(), True),
        StructField("STUDENT_ID", LongType(), True),
        StructField("RAW_MARK", StringType(), True),
        StructField("MARK_CODE", StringType(), True),
        StructField("COURSE_UNIQUE_ID", StringType(), True),
        StructField("COURSE_ID", LongType(), True),
        StructField("LESSON_UNIQUE_ID", StringType(), True),
        StructField("LESSON_ID", LongType(), True),
        StructField("START_DATE_TIME", TimestampType(), True),
        StructField("END_DATE_TIME", TimestampType(), True),
        StructField("MINUTES_LATE", LongType(), True),
        StructField("IS_PRESENT", BooleanType(), True),
        StructField("IS_AUTHORIZED_ABSENT", BooleanType(), True),
        StructField("IS_UNAUTHORIZED_ABSENT", BooleanType(), True),
        StructField("IS_POSSIBLE_ATTENDANCE", BooleanType(), True)
    ]),
    "COURSES" :StructType([
        StructField("COURSE_UNIQUE_ID", StringType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("COURSE_ID", LongType(), True),
        StructField("PARENT_COURSE_ID", LongType(), True),
        StructField("PARENT_COURSE_UNIQUE_ID", StringType(), True),
        StructField("COURSE_CODE", StringType(), True),
        StructField("COURSE_NAME", StringType(), True),
        StructField("ACADEMIC_YEAR_ID", LongType(), True),
        StructField("ACADEMIC_YEAR_UNIQUE_ID", StringType(), True),
        StructField("ACADEMIC_LEVEL_ID", LongType(), True),
        StructField("ACADEMIC_LEVEL_UNIQUE_ID", StringType(), True),
        StructField("FACULTY_ID", LongType(), True),
        StructField("FACULTY_UNIQUE_ID", StringType(), True),
        StructField("SUBJECT_ID", LongType(), True),
        StructField("SUBJECT_UNIQUE_ID", StringType(), True),
        StructField("SUBJECT_CODE", StringType(), True),
        StructField("SUBJECT_NAME", StringType(), True),
        StructField("IS_MAIN_ASSESSABLE_UNIT", BooleanType(), True),
        StructField("DEPARTMENT_ID", StringType(), True),
        StructField("DEPARTMENT_UNIQUE_ID", StringType(), True),
        StructField("IS_CLASS", BooleanType(), True)
    ]),
    "COURSE_ENROLMENTS" : StructType([
        StructField("COURSE_ENROLMENT_UNIQUE_ID", StringType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("COURSE_ENROLMENT_ID", LongType(), True),
        StructField("COURSE_ID", LongType(), True),
        StructField("COURSE_UNIQUE_ID", StringType(), True),
        StructField("STUDENT_ID", LongType(), True),
        StructField("STUDENT_UNIQUE_ID", StringType(), True),
        StructField("START_DATE", DateType(), True),
        StructField("END_DATE", DateType(), True),
        StructField("ENROLMENT_STATUS", StringType(), True),
        StructField("REPEAT_ENROLMENT", BooleanType(), True)
    ]),
    "COURSE_LEADS" : StructType([
        StructField("COURSE_LEAD_UNIQUE_ID", StringType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("COURSE_LEAD_ID", LongType(), True),
        StructField("COURSE_UNIQUE_ID", StringType(), True),
        StructField("COURSE_ID", LongType(), True),
        StructField("STAFF_UNIQUE_ID", StringType(), True),
        StructField("STAFF_ID", LongType(), True),
        StructField("START_DATE", DateType(), True),
        StructField("END_DATE", DateType(), True)
    ]),
    "STUDENT_SCHOOL_ENROLMENTS" :StructType([
        StructField("STUDENT_UNIQUE_ID", StringType(), True),                  
        StructField("STUDENT_SCHOOL_ENROLMENT_ID", LongType(), True),         
        StructField("STUDENT_SCHOOL_ENROLMENT_UNIQUE_ID", StringType(), True), 
        StructField("APPLICATION_ID", StringType(), True),                     
        StructField("STUDENT_ID", LongType(), True),                           
        StructField("START_DATE", DateType(), True),                        
        StructField("END_DATE", DateType(), True),                          
        StructField("CURRENTLY_ON_ROLL", BooleanType(), True),               
        StructField("UNENROLMENT_REASON_CODE", StringType(), True)             
    ]),
    "ACADEMIC_YEARS" : StructType([
        StructField("ACADEMIC_YEAR_UNIQUE_ID", StringType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("ACADEMIC_YEAR_NAME", StringType(), True),
        StructField("CALENDAR_YEAR_START_DATE", DateType(), True),
        StructField("CALENDAR_YEAR_END_DATE", DateType(), True),
        StructField("ACADEMIC_YEAR_CODE", StringType(), True),
        StructField("IS_CURRENT", BooleanType(), True)
    ]),
    "BEHAVIOURAL_INCIDENT_ACTIONS": StructType([
        StructField("BEHAVIOURAL_INCIDENT_ACTION_UNIQUE_ID", StringType(), True),
        StructField("BEHAVIOURAL_INCIDENT_ACTION_ID", FloatType(), True),
        StructField("BEHAVIOURAL_INCIDENT_ID", FloatType(), True),
        StructField("BEHAVIOURAL_INCIDENT_UNIQUE_ID", StringType(), True),
        StructField("STUDENT_ID", FloatType(), True),
        StructField("STUDENT_UNIQUE_ID", StringType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("POINT_AWARD_UNIQUE_ID", StringType(), True),
        StructField("DETENTION_UNIQUE_ID", StringType(), True),
        StructField("INTERNAL_EXCLUSION_UNIQUE_ID", StringType(), True),
        StructField("SUSPENSION_UNIQUE_ID", StringType(), True),
        StructField("PERMANENT_EXCLUSION_UNIQUE_ID", StringType(), True),
    ]),
    "DETENTIONS": StructType([
        StructField("DETENTION_UNIQUE_ID", StringType(), True),
        StructField("DETENTION_ID", IntegerType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("STUDENT_UNIQUE_ID", StringType(), True),
        StructField("STUDENT_ID", IntegerType(), True),
        StructField("ISSUED_BY_STAFF_UNIQUE_ID", StringType(), True),
        StructField("ISSUED_BY_STAFF_ID", IntegerType(), True),
        StructField("DETENTION_SESSION_UNIQUE_ID", StringType(), True),
        StructField("DETENTION_SESSION_ID", IntegerType(), True),
        StructField("DETENTION_TYPE", StringType(), True),
        StructField("START_DATETIME", TimestampNTZType(), True),
        StructField("END_DATETIME", TimestampNTZType(), True),
        StructField("ATTENDANCE_MARK", StringType(), True),
        StructField("REASON_FOR_DETENTION_UNIQUE_ID", StringType(), True),
        StructField("REASON_FOR_DETENTION_ID", IntegerType(), True),
        StructField("REASON_FOR_DETENTION_BEHAVIOUR_NAME", StringType(), True),
        StructField("ROOM_UNIQUE_ID", StringType(), True),
        StructField("ROOM_ID", IntegerType(), True),
        StructField("ACADEMIC_YEAR_UNIQUE_ID", StringType(), True),
        StructField("ACADEMIC_YEAR_ID", IntegerType(), True),
        StructField("ACADEMIC_YEAR_NAME", StringType(), True),
        StructField("DECISION_DATETIME", TimestampNTZType(), True),
    ]),
    "POINT_AWARDS": StructType([
        StructField("POINT_AWARD_ID", IntegerType(), True),
        StructField("POINT_AWARD_UNIQUE_ID", StringType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("STUDENT_UNIQUE_ID", StringType(), True),
        StructField("STUDENT_ID", IntegerType(), True),
        StructField("POINT_AWARD_CATEGORY", StringType(), True),
        StructField("POINTS", FloatType(), True),
        StructField("AWARDED_DATE", DateType(), True),
        StructField("AWARDED_BY_STAFF_ID", IntegerType(), True),
        StructField("AWARDED_BY_STAFF_UNIQUE_ID", StringType(), True),
        StructField("NARRATIVE", StringType(), True),
        StructField("POINT_AWARD_SCALE", StringType(), True),
        StructField("POINT_AWARD_SCALE_ID", IntegerType(), True),
        StructField("POINT_AWARD_SCALE_UNIQUE_ID", StringType(), True),
        StructField("ROOM_UNIQUE_ID", StringType(), True),
        StructField("LESSON_UNIQUE_ID", StringType(), True),
        StructField("ACADEMIC_YEAR_UNIQUE_ID", StringType(), True),
        StructField("ACADEMIC_YEAR_ID", IntegerType(), True),
        StructField("ACADEMIC_YEAR_NAME", StringType(), True),
    ]),
    "INTERNAL_EXCLUSIONS": StructType([
        StructField("INTERNAL_EXCLUSION_UNIQUE_ID", StringType(), True),
        StructField("INTERNAL_EXCLUSION_ID", IntegerType(), True),
        StructField("APPLICATION_ID", StringType(), True),
        StructField("STUDENT_UNIQUE_ID", StringType(), True),
        StructField("STUDENT_ID", IntegerType(), True),
        StructField("ISSUED_DATETIME", TimestampNTZType(), True),
        StructField("ISSUED_BY_STAFF_UNIQUE_ID", StringType(), True),
        StructField("ISSUED_BY_STAFF_ID", IntegerType(), True),
        StructField("INTERNAL_EXCLUSION_REASONS_NAME", StringType(), True),
        StructField("INTERNAL_EXCLUSION_REASONS_TYPE_NAME", StringType(), True),
        StructField("NARRATIVE", StringType(), True),
        StructField("TIMETABLE_SLOT_UNIQUE_ID", StringType(), True),
        StructField("TIMETABLE_SLOT_ID", IntegerType(), True),
        StructField("LOCATION_UNIQUE_ID", StringType(), True),
        StructField("LOCATION_ID", IntegerType(), True),
        StructField("SESSION_NAME", StringType(), True),
        StructField("START_DATETIME", TimestampNTZType(), True),
        StructField("END_DATETIME", TimestampNTZType(), True),
        StructField("RAW_MARK", StringType(), True),
        StructField("MARK_CODE", StringType(), True),
        StructField("MINUTES_LATE", IntegerType(), True),
        StructField("IS_PRESENT", BooleanType(), True),
        StructField("IS_AUTHORIZED_ABSENT", BooleanType(), True),
        StructField("IS_UNAUTHORIZED_ABSENT", BooleanType(), True),
        StructField("IS_POSSIBLE_ATTENDANCE", BooleanType(), True),
    ]),
}

In [ ]:
# Define large tables to handle with pagination
large_tables = ["ROLL_CALL_ATTENDANCE", "LESSON_ATTENDANCE"]

In [ ]:
oeai.log.start_block("Functions")

In [ ]:
def fetch_data_in_chunks(query, view_name, start_date, num_days=7):
    current_date = start_date
    today = datetime.today()
    first_write = True

    while current_date < today:
        end_date = current_date + timedelta(days=num_days)
        if end_date > today:
            end_date = today

        # Adjust end_date to be exclusive
        adjusted_end_date = end_date - timedelta(seconds=1)

        paginated_query = query.format(start_date=current_date.strftime('%Y-%m-%d'), end_date=adjusted_end_date.strftime('%Y-%m-%d'))
        # print(f"Executing query for {view_name} from {current_date} to {adjusted_end_date}")
        oeai.log.info(f"Executing query for {view_name} from {current_date} to {adjusted_end_date}", view_name=view_name, start_date=current_date, end_date=adjusted_end_date)
        
        with conn.cursor() as cur:
            cur.execute(paginated_query)
            df_pandas = cur.fetch_pandas_all()
            if df_pandas.empty:
                # print(f"No more data to fetch for {view_name} from {current_date} to {adjusted_end_date}. Moving to next date range.")
                oeai.log.debug(f"No more data to fetch for {view_name} from {current_date} to {adjusted_end_date}. Moving to next date range.", view_name=view_name, start_date=current_date, end_date=adjusted_end_date)
                current_date = end_date + timedelta(days=1)
                continue

            df_spark = spark.createDataFrame(df_pandas, schema=schemas[view_name])
            write_mode = "overwrite" if first_write else "append"

            df_spark.write.format("parquet") \
                .mode(write_mode) \
                .partitionBy("Application_ID") \
                .save(f"{bronze_path}/{view_name}")
            
            if first_write:
                first_write = False

            # print(f"Fetched and saved data for {view_name} from {current_date} to {end_date}")
            oeai.log.info(f"Fetched and saved data for {view_name} from {current_date} to {end_date}")
        
        current_date = end_date #+ timedelta(days=1)


In [ ]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.checkpoint("Init")

In [ ]:
oeai.log.start_block("Processing")

from datetime import datetime, timedelta
from json.decoder import JSONDecodeError

# Process each query
with snowflake.connector.connect(**conn_params) as conn:
    for query, view_name in queries:
        # print(f"\nStarting to process view: {view_name}")
        oeai.log.start_block(block_type="Query", block_name=view_name)
        oeai.log.info(f"Starting to process view: {view_name}", view_name=view_name)
        
        if view_name in large_tables:
            print(start_date)
            oeai.log.debug(start_date, start_date=start_date)
            # Fetch data incrementally by date
            fetch_data_in_chunks(query, view_name, start_date, num_days)

        else:
            # Execute query for the entire date range
            end_date = datetime.today().strftime('%Y-%m-%d')
            query = query.format(start_date=start_date)
                        
            with conn.cursor() as cur:
                # print(f"Executing query for {view_name}")
                oeai.log.info(f"Executing query for {view_name}", view_name=view_name)

                cur.execute(query)
                # Fetch results into Pandas DataFrame
                df_pandas = cur.fetch_pandas_all()
            # print(f"Fetched data for {view_name}, converting to Spark DataFrame")
            oeai.log.info(f"Fetched data for {view_name}, converting to Spark DataFrame", view_name=view_name)

            # Handle NaN values based on column data type
            for column in df_pandas.columns:
                if pd.api.types.is_integer_dtype(df_pandas[column]):
                    df_pandas[column].fillna(0, inplace=True)
                elif pd.api.types.is_float_dtype(df_pandas[column]):
                    df_pandas[column].fillna(0.0, inplace=True)
                elif pd.api.types.is_string_dtype(df_pandas[column]):
                    df_pandas[column].fillna('', inplace=True)
                elif pd.api.types.is_datetime64_any_dtype(df_pandas[column]):
                    df_pandas[column].fillna(pd.Timestamp('1970-01-01'), inplace=True)
                elif pd.api.types.is_bool_dtype(df_pandas[column]):
                    df_pandas[column].fillna(False, inplace=True)

            # 🚀 Step 1: Identify Columns That Need Processing

            date_columns = [field.name for field in schemas[view_name] if isinstance(field.dataType, DateType)]
            print(f"📌 Identified Date Columns for {view_name}: {date_columns}")
            
            timestamp_columns = [field.name for field in schemas[view_name] if isinstance(field.dataType, TimestampNTZType)]
            print(f"📌 Identified Timestamp Columns for {view_name}: {timestamp_columns}")
            
            integer_columns = [field.name for field in schemas[view_name] if isinstance(field.dataType, IntegerType)]
            print(f"📌 Identified Integer Columns for {view_name}: {integer_columns}")
            
            float_columns = [field.name for field in schemas[view_name] if isinstance(field.dataType, FloatType)]
            print(f"📌 Identified Float Columns for {view_name}: {float_columns}")


            # 🚀 Step 2: Convert Each Column to the Correct Data Type
            for col in date_columns:
                if col in df_pandas.columns:
                    print(f"\n🚀 Processing Date Column: {col}")
                    df_pandas[col] = pd.to_datetime(df_pandas[col], errors='coerce').dt.date  # Keep only the date
                    print(f"✅ {col} successfully converted to DateType")

            for col in timestamp_columns:
                if col in df_pandas.columns:
                    print(f"\n🚀 Processing Timestamp Column: {col}")
                    df_pandas[col] = pd.to_datetime(df_pandas[col], errors='coerce')  # Keep full timestamp
                    print(f"✅ {col} successfully converted to TimestampNTZType")

            for col in integer_columns:
                if col in df_pandas.columns:
                    print(f"\n🚀 Processing Integer Column: {col}")
                    df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce').astype("Int64")  # Enforce integer
                    print(f"✅ {col} successfully converted to IntegerType")

            for col in float_columns:
                if col in df_pandas.columns:
                    print(f"\n🚀 Processing Float Column: {col}")
                    df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce').astype("float")  # Enforce float
                    print(f"✅ {col} successfully converted to FloatType")

            # 🚀 Step 3: Ensure No Empty Strings Exist Before Passing to Spark
            df_pandas.replace(['', 'NULL', 'None'], None, inplace=True)

            # Convert Pandas DataFrame to Spark DataFrame using the specific schema
            df_spark = spark.createDataFrame(df_pandas, schema=schemas[view_name])
            
            output_path = bronze_path + view_name
            
            # Write data to Parquet, naming directory after the view
            df_spark.write.format("parquet") \
                .mode("overwrite") \
                .partitionBy("Application_ID") \
                .save(output_path)
                
            # print(f"Completed processing for {view_name}")
            oeai.log.info(f"Completed processing for {view_name}", view_name=view_name)
            oeai.log.end_block(index=2, include_target=True)
            oeai.log.checkpoint(view_name)

oeai.log.end_block(index=1, include_target=True)
# print("All queries processed.")
oeai.log.info("All queries processed")
oeai.log.end_block(index=0, include_target=True)
oeai.log.checkpoint("Processing")

In [ ]:
# Close the connection after all queries are processed
conn.close()

In [ ]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.shutdown()
oeai.log.checkpoint()